# MTX lda Emails

**Status:** Versão de referência · **Área:** topicos-emails

Variante LDA com busca de coerência, tokenizador NLTK e spaCy; saída contém falta de punkt_tab.

Este arquivo está identificado como **Versão de referência**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


In [ ]:
# Instalação de dependências (se necessário)
!pip install gensim numpy==1.24.4 pyLDAvis unidecode nltk spacy tqdm --quiet
# Reinicie o runtime após instalar, caso esteja em ambiente como Colab


In [ ]:
import pandas as pd
import numpy as np
import re, string, unidecode, warnings
from tqdm import tqdm
import nltk
import gensim
from gensim.models import Phrases, LdaModel
from gensim.models.phrases import Phraser
from gensim.corpora import Dictionary
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
warnings.filterwarnings('ignore')
print('pandas:', pd.__version__)
print('gensim:', gensim.__version__)


In [ ]:
# Recursos NLTK e spaCy
nltk.download('punkt')
nltk.download('stopwords')
# Modelo spaCy para PT
!python -m spacy download pt_core_news_sm --quiet
nlp = spacy.load('pt_core_news_sm')


In [ ]:
# ↓ Rode esta célula só uma vez para preparar o ambiente
import nltk, spacy, subprocess, sys

# Baixar tokenizador e stopwords do NLTK
nltk.download('punkt')
nltk.download('stopwords')

# Garantir que o modelo de português do spaCy esteja presente
try:
    nlp = spacy.load('pt_core_news_sm')
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "pt_core_news_sm"])
    nlp = spacy.load('pt_core_news_sm')


In [ ]:
# Carregar dataset de e‑mails
df = pd.read_csv('/content/dataset_emails.csv')
# Unir assunto + corpo caso deseje
df['texto'] = df['assunto'].fillna('') + ' ' + df['corpo'].fillna('')
data = df[['texto']].rename(columns={'texto':'description'})
print(f'{data.shape[0]} e‑mails carregados')
data.head()

In [ ]:
###############################
# Funções de limpeza
###############################
stop_words = set(stopwords.words('portuguese'))
def clean_text(text):
    text = text.lower()
    text = unidecode.unidecode(text)
    text = ''.join(ch for ch in text if not ch.isdigit())
    text = ''.join(ch for ch in text if ch not in string.punctuation)
    text = ' '.join(text.split())
    return text

def clean_text_by_pos(text, filterPOS=True, allowed_postags=['NOUN','PROPN','VERB','ADJ','ADV'], lemma=False):
    doc = nlp(text)
    if filterPOS:
        if lemma:
            cleaned = ' '.join([token.lemma_ if token.pos_!='PROPN' else token.text for token in doc if token.pos_ in allowed_postags])
        else:
            cleaned = ' '.join([token.text for token in doc if token.pos_ in allowed_postags])
    else:
        if lemma:
            cleaned = ' '.join([token.lemma_ for token in doc])
        else:
            cleaned = text
    return cleaned

def preprocess_text(text, filterPOS=True, lemma=True, allowed_postags=['NOUN','PROPN','VERB','ADJ','ADV'], min_word_len=4):
    text = clean_text_by_pos(text, filterPOS=filterPOS, lemma=lemma, allowed_postags=allowed_postags)
    text = clean_text(text)
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words and len(w)>=min_word_len]
    return words


In [ ]:
allowed_postags = ['NOUN','PROPN','VERB','ADJ','ADV']
tqdm.pandas(desc='Pré-processando')
data['processed'] = data['description'].progress_apply(lambda x: preprocess_text(x, allowed_postags=allowed_postags))
data.head()

In [ ]:
# Bigrams / Trigrams
bigrams = Phrases(data['processed'], min_count=1, threshold=2)
bigram_mod = Phraser(bigrams)
data['processed'] = data['processed'].apply(lambda x: bigram_mod[x])
data['processed'].head()

In [ ]:
dictionary = Dictionary(data['processed'])
dictionary.filter_extremes(no_below=1, no_above=0.8)
corpus = [dictionary.doc2bow(doc) for doc in data['processed']]
print(f'Dicionário com {len(dictionary)} tokens')

In [ ]:
# Treinar LDA
num_topics = 8
lda_model = LdaModel(corpus=corpus, num_topics=num_topics, id2word=dictionary, passes=100, random_state=42)
print('Modelo treinado')

In [ ]:
topics = lda_model.print_topics(num_words=6)
for t in topics:
    print(f'Tópico {t[0]}: {t[1]}')

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

def compute_coherence_perplexity(dictionary, corpus, texts, limit, passes=250, iterations=150, start=2, step=1):
    coherence_values = []
    perplexity_values = []
    model_list = []
    for num_topics in range(start, limit, step):
        model = LdaModel(corpus=corpus, num_topics=num_topics, id2word=dictionary, passes=passes, iterations=iterations, random_state=42)
        model_list.append(model)
        coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_values.append(coherencemodel.get_coherence())
        perplexity_values.append(model.log_perplexity(corpus))
    # Melhor número de tópicos
    best_idx = np.argmax(coherence_values)
    best_num_topics = range(start, limit, step)[best_idx]
    return coherence_values, perplexity_values, best_num_topics


In [ ]:
%%time
coherence_values, perplexity_values, best_num_topics = compute_coherence_perplexity(dictionary, corpus, data['processed'], limit=11)
print('Melhor número estimado de tópicos:', best_num_topics)
print('Coherence values:', coherence_values)
print('Perplexity values:', perplexity_values)

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
pyLDAvis.enable_notebook()
lda_vis = gensimvis.prepare(lda_model, corpus, dictionary)
lda_vis

## Continuidade

Consolidar recursos linguísticos e versões; treinar modelo final com o melhor k escolhido (hoje busca e treino fixo não estão conectados); preservar função útil e arquivar redundância.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.